<a href="https://colab.research.google.com/github/41371204h/1142-programming-language/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-generativeai

In [50]:
# 安裝 PDF 生成庫
!pip install reportlab
!apt-get -y install fonts-wqy-zenhei # 確保中文字體已安裝

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-wqy-zenhei is already the newest version (0.9.45-8).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.


In [ ]:
import os
import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import gspread
import gradio as gr
from datetime import datetime
from google.colab import auth, userdata
from google.auth import default
from google import genai

In [51]:
# --- PDF 生成相關套件 ---
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.units import cm
from reportlab.lib.utils import ImageReader
from reportlab.lib.colors import navy, red, black, green

In [ ]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1bMteJ88u-_o-6v1Ym1RZNDrdIpQA3_A6OucFpDQbyBs/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [52]:
# --- 1. 設定中文字體 (Colab 環境) ---
font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
if os.path.exists(font_path):
    # 用於 Matplotlib
    import matplotlib.font_manager as fm
    fe = fm.FontEntry(fname=font_path, name='ZenHei')
    fm.fontManager.ttflist.insert(0, fe)
    plt.rcParams['font.family'] = fe.name
    plt.rcParams['axes.unicode_minus'] = False # 修正負號顯示問題

    # 用於 ReportLab (PDF)
    pdfmetrics.registerFont(TTFont('ZenHei', font_path))
else:
    print("⚠️ 警告：找不到中文字體，PDF 和圖表可能出現亂碼。")

In [ ]:
# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
response = client.models.generate_content(
    model = MODEL_ID, contents="Explain how AI works in a few words"
)
print(response.text)

AI teaches computers to **learn from data**, **recognize patterns**, and then **make decisions or predictions**.


In [25]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print(f"\n--- [DEBUG] 準備呼叫 AI 模型... ---\n提示長度: {len(prompt_text)} 字符")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        print("--- [DEBUG] AI 模型回應成功。---")
        # 將回傳內容中的 * 替換為空白
        summary = response.text.replace('*', ' ')
        return summary
    except Exception as e:
        print(f"--- [ERROR] 呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"


In [53]:
def draw_radar_chart(subjects, scores, avg_score):
    """
    繪製科目平衡雷達圖。
    """
    num_vars = len(subjects)

    # 計算角度，並閉合圖形 (重複第一個點)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    scores = scores + [scores[0]]
    angles = angles + [angles[0]]
    plot_subjects = subjects + [subjects[0]]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    # 繪製背景網格與科目名稱
    ax.set_theta_offset(np.pi / 2) # 設定頂點為起始點
    ax.set_theta_direction(-1)     # 順時針方向
    plt.xticks(angles[:-1], subjects, color='grey', size=12)

    # 繪製 Y 軸 (分數)
    ax.set_rlabel_position(0)
    plt.yticks([20, 40, 60, 80, 100], ["20", "40", "60", "80", "100"], color="grey", size=10)
    plt.ylim(0, 100)

    # --- 繪製成績區域 ---
    ax.plot(angles, scores, linewidth=2, linestyle='solid', color='#3498db', label='本次表現')
    ax.fill(angles, scores, color='#3498db', alpha=0.25)

    # --- 繪製平均分數圓圈 (參考線) ---
    ax.plot(angles, [avg_score] * (num_vars + 1), linewidth=1.5, linestyle='--', color='#e74c3c', label=f'平均線 ({avg_score:.1f})')

    # --- 繪製及格圓圈 (60分) ---
    ax.plot(angles, [60] * (num_vars + 1), linewidth=1, linestyle=':', color='#95a5a6', label='及格線')

    plt.title(f"科目平衡度分析雷達圖", size=16, color='#2c3e50', y=1.1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

    return fig

In [54]:
def generate_pdf_report(today, subjects, scores, stats_info, ai_summary, fig):
    """
    將統計數據、AI 摘要和圖表整合成 PDF 報告。
    """
    filename = f"成績分析報告_{today}.pdf"
    c = canvas.Canvas(filename, pagesize=A4)
    width, height = A4

    # 設定字體
    c.setFont('ZenHei', 20)

    # 1. 頁眉
    c.setFillColor(navy)
    c.drawString(2*cm, height - 2*cm, f"📊 學生個人成績分析報告")
    c.setFont('ZenHei', 12)
    c.setFillColor(black)
    c.drawString(2*cm, height - 2.8*cm, f"報告日期：{today}")
    c.line(2*cm, height - 3*cm, width - 2*cm, height - 3*cm)

    # 2. 數據統計區塊
    c.setFont('ZenHei', 14)
    c.setFillColor(navy)
    c.drawString(2*cm, height - 4*cm, "一、 數據總結")

    c.setFont('ZenHei', 11)
    c.setFillColor(black)
    text_y = height - 4.8*cm

    # 解析統計資訊字串 (從 stats_info 中提取)
    for line in stats_info.split('\n'):
        if line.strip():
            c.drawString(2.5*cm, text_y, line)
            text_y -= 0.6*cm

    # 3. 插入雷達圖
    c.setFont('ZenHei', 14)
    c.setFillColor(navy)
    c.drawString(2*cm, text_y - 1*cm, "二、 科目平衡度分析圖")

    # 將 Matplotlib 圖表轉為圖片數據
    img_buffer = io.BytesIO()
    fig.savefig(img_buffer, format='png', bbox_inches='tight', dpi=150)
    img_buffer.seek(0)

    # 插入圖片 (調整位置和大小)
    c.drawImage(ImageReader(img_buffer), 4*cm, text_y - 11*cm, width=13*cm, height=10*cm, mask='auto')

    # 4. AI 摘要區塊 (需處理自動換行)
    text_y_ai = text_y - 12*cm
    c.setFont('ZenHei', 14)
    c.setFillColor(navy)
    c.drawString(2*cm, text_y_ai, "三、 🤖 AI 學習建議與迷思診斷")

    c.setFont('ZenHei', 11)
    c.setFillColor(black)
    text_y_content = text_y_ai - 0.8*cm

    # 簡單的自動換行邏輯
    max_char_per_line = 35 # 每行最大中文字數

    # 先依照 AI 摘要原本的換行符號拆分
    for paragraph in ai_summary.split('\n'):
        if not paragraph.strip(): # 處理空行
            text_y_content -= 0.5*cm
            continue

        # 針對長段落進行換行
        for i in range(0, len(paragraph), max_char_per_line):
            line_content = paragraph[i:i+max_char_per_line]
            c.drawString(2.5*cm, text_y_content, line_content)
            text_y_content -= 0.6*cm

            # 如果快超出頁面，則新建一頁 (簡化版，這裡先假設只有一頁)
            if text_y_content < 2*cm:
                c.showPage()
                c.setFont('ZenHei', 11) # 新頁需重新設定字體
                text_y_content = height - 2*cm

    c.save()
    return filename

In [57]:
def process_grades_and_summary(grade_data):
    """
    核心邏輯：處理成績、繪製雷達圖、生成 PDF 並寫入 Google Sheet。
    """
    global _gc, _ws

    # 檢查連線 (這裡假設 setup_gspread 已經在外部被正確呼叫)
    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            return "Google Sheet 連線失敗", "", None, None

    # 1. 資料清理與轉換
    # 這次強化了空白行過濾，確保 type 為 'array' 時的穩定性
    if grade_data is None:
        return "請輸入資料。", "", None, None

    valid_data = []
    if isinstance(grade_data, pd.DataFrame):
        valid_data = grade_data.dropna().values.tolist()
    else:
        # 假設是列表的列表 (array type)
        valid_data = [row for row in grade_data if len(row) >= 2 and row[0] and str(row[1]).strip()]

    if not valid_data:
        return "請至少輸入一個科目的有效成績。", "", None, None

    subjects = []
    scores = []
    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')

    for subject, grade_str in valid_data:
        try:
            score = int(float(grade_str)) # 先轉 float 再轉 int，容錯率更高
            if score < 0 or score > 100:
                return f"科目 '{subject}' 的成績 ({score}) 超出範圍 (0-100)。", "", None, None
            subjects.append(subject)
            scores.append(score)
            new_grades_for_sheet.append([today, subject, score])
        except ValueError:
            return f"科目 '{subject}' 的成績格式錯誤，請輸入數字。", "", None, None

    # 雷達圖至少需要 3 個科目才能畫出「面」
    if len(subjects) < 3:
        return "雷達圖分析至少需要輸入 3 個科目。", "", None, None

    # 2. 數據計算與圖表繪製
    avg_score = sum(scores) / len(scores)
    failed_count = sum(1 for s in scores if s < 60)
    best_subject = subjects[scores.index(max(scores))]

    # 繪製雷達圖 (取代之前的長條圖和折線圖)
    fig = draw_radar_chart(subjects, scores, avg_score)

    # 3. 取得 AI 摘要與建立文字總結
    ai_raw_summary = get_ai_summary(new_grades_for_sheet) # 使用原本的 get_ai_summary

    stats_summary = (
        f"📊 【本次表現硬數據】\n"
        f"● 科目總數：{len(subjects)}\n"
        f"● 平均分數：{avg_score:.2f} 分\n"
        f"● 表現最優：{best_subject} ({max(scores)}分)\n"
        f"● 未及格科目：{failed_count} 科"
    )

    # 介面顯示用的完整摘要
    display_summary = f"{stats_summary}\n\n--------------------------------\n\n🤖 【AI 深度分析建議】\n{ai_raw_summary}"

    # 4. 生成 PDF 報告檔案
    pdf_file_path = generate_pdf_report(today, subjects, scores, stats_summary, ai_raw_summary, fig)

    # 5. 寫入 Google Sheet (精簡版總結)
    sheet_message = ""
    try:
        # 先寫入明細成績
        _ws.append_rows(new_grades_for_sheet)

        # 再寫入一列精簡總結
        summary_row = [
            today,
            f"總結(平均:{avg_score:.1f}|最優:{best_subject})",
            ai_raw_summary
        ]
        _ws.append_row(summary_row)
        sheet_message = f"✅ 成功！資料已寫入 Sheet，平均分 {avg_score:.1f}。"
    except Exception as e:
        sheet_message = f"❌ Sheet 寫入失敗: {e}"

    # 返回：Gradio 訊息, 文字摘要, 雷達圖, PDF 檔案路徑
    return sheet_message, display_summary, fig, pdf_file_path

In [58]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 專業成績分析報告系統 (雷達圖 & PDF 版)")
    gr.Markdown("請輸入至少 3 個科目的成績，系統會生成平衡度雷達圖、AI 建議與 PDF 報告，並同步上傳至 Google Sheet。")

    with gr.Row():
        with gr.Column(scale=2):
            # 設定預設資料，方便測試
            default_data = [["國文", "92"], ["數學", "65"], ["英文", "88"], ["物理", "70"], ["歷史", "95"]]
            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=default_data,
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="輸入區域 (科目與分數)"
            )

            with gr.Row():
                submit_button = gr.Button("🚀 開始分析並生成報告", variant="primary")
                clean_button = gr.Button("🗑️ 清空資料")

            sheet_output = gr.Textbox(label="系統處理訊息")

            # --- 新增 PDF 下載元件 ---
            pdf_download = gr.File(label="📥 下載完整 PDF 分析報告", file_types=[".pdf"])

        with gr.Column(scale=3):
            # 顯示雷達圖
            plot_output = gr.Plot(label="科目平衡度分析雷達圖")
            # 顯示 AI 摘要
            summary_output = gr.Textbox(label="數據總結與 AI 建議", lines=15)

    # 按鈕互動邏輯
    submit_button.click(
        process_grades_and_summary,
        inputs=grade_input,
        outputs=[sheet_output, summary_output, plot_output, pdf_download] # 確保有 4 個輸出元件
    )

    # 清空按鈕邏輯
    clean_button.click(
        lambda: ([["", ""]], "", None, None),
        outputs=[grade_input, sheet_output, plot_output, pdf_download]
    )

# launch 前請確認 setup_gspread 已經被呼叫過
# setup_gspread(SHEET_URL, WORKSHEET_NAME)
demo.launch()

/tmp/ipykernel_25277/900273533.py:1: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a709595798cd9f3201.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
